# 13 · Random Forest Classification
Train and evaluate Random Forest classifiers on UrbanSound8K training folds (1–9) with 9-fold grouped CV (LeaveOneGroupOut), tune n_estimators, evaluate on held-out test Fold 10, compare against SVM candidate, and inspect top feature importances.

In [1]:
# Cell 1 – interpreter setup & imports
import sys
from pathlib import Path
import time
import hashlib

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

print('Successfully imported sklearn, joblib, numpy, pandas, and matplotlib.pyplot')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported sklearn, joblib, numpy, pandas, and matplotlib.pyplot


In [2]:
# Cell 2 – verify sha256 of inputs, load data and build train/test sets
def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

labels_path = REPO_ROOT / 'data/processed/labels.csv'
features_path = REPO_ROOT / 'data/processed/audio_features.npy'
split_path = REPO_ROOT / 'data/processed/split_indices.npz'

labels_sha = compute_sha256(labels_path)
features_sha = compute_sha256(features_path)
split_sha = compute_sha256(split_path)

print(f'labels.csv sha256        : {labels_sha}')
print(f'audio_features.npy sha256: {features_sha}')
print(f'split_indices.npz sha256 : {split_sha}')

EXPECTED_LABELS_SHA = '8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854'
EXPECTED_FEAT_SHA   = 'ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a'
EXPECTED_SPLIT_SHA  = '2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b'

assert labels_sha == EXPECTED_LABELS_SHA, f'labels.csv SHA mismatch: {labels_sha}'
assert features_sha == EXPECTED_FEAT_SHA, f'audio_features.npy SHA mismatch: {features_sha}'
assert split_sha == EXPECTED_SPLIT_SHA, f'split_indices.npz SHA mismatch: {split_sha}'

labels = pd.read_csv(labels_path)
X = np.load(features_path)

with np.load(split_path) as split_data:
    train_idx = split_data['train_idx']
    test_idx = split_data['test_idx']
    esc_idx = split_data['esc_idx']

X_train = X[train_idx]
y_train = labels.loc[train_idx, 'app_class'].to_numpy()
groups_train = labels.loc[train_idx, 'fold'].to_numpy()

X_test = X[test_idx]
y_test = labels.loc[test_idx, 'app_class'].to_numpy()

print(f'X_train shape     : {X_train.shape}')
print(f'y_train shape     : {y_train.shape}')
print(f'groups_train shape: {groups_train.shape}')
print(f'X_test shape      : {X_test.shape}')
print(f'y_test shape      : {y_test.shape}')

assert X_train.shape == (5753, 128), f'Expected X_train shape (5753, 128), got {X_train.shape}'
assert X_test.shape == (605, 128), f'Expected X_test shape (605, 128), got {X_test.shape}'
assert len(y_train) == 5753, f'Expected 5753 in y_train, got {len(y_train)}'
assert len(groups_train) == 5753, f'Expected 5753 in groups_train, got {len(groups_train)}'
assert len(y_test) == 605, f'Expected 605 in y_test, got {len(y_test)}'

print('Cell 2 SHA256 asserts and loads passed ✓')

labels.csv sha256        : 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
audio_features.npy sha256: ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a
split_indices.npz sha256 : 2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b
X_train shape     : (5753, 128)
y_train shape     : (5753,)
groups_train shape: (5753,)
X_test shape      : (605, 128)
y_test shape      : (605,)
Cell 2 SHA256 asserts and loads passed ✓


In [3]:
# Cell 3 – evaluate Random Forest with 200 and 300 estimators via LeaveOneGroupOut CV
cv = LeaveOneGroupOut()
rf_results = {}

for n in [200, 300]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1, class_weight='balanced')
    t0 = time.perf_counter()
    scores = cross_val_score(rf, X_train, y_train, groups=groups_train, cv=cv, scoring='accuracy')
    elapsed = time.perf_counter() - t0
    mean_acc = float(np.mean(scores))
    std_acc = float(np.std(scores))
    rf_results[n] = {
        'mean': mean_acc,
        'std': std_acc,
        'scores': scores,
        'time_s': elapsed
    }
    print(f'n_estimators={n:3d} | CV Accuracy: {mean_acc:.4f} (+/- {std_acc:.4f}) | Time: {elapsed:.2f} s')

best_n = max(rf_results, key=lambda k: rf_results[k]['mean'])
best_rf_cv_mean = rf_results[best_n]['mean']
best_rf_cv_std = rf_results[best_n]['std']

print(f'\nSelected n_estimators={best_n} with higher mean CV accuracy: {best_rf_cv_mean:.4f} (+/- {best_rf_cv_std:.4f})')
print('Cell 3 completed ✓')

n_estimators=200 | CV Accuracy: 0.6856 (+/- 0.0546) | Time: 18.44 s


n_estimators=300 | CV Accuracy: 0.6864 (+/- 0.0568) | Time: 28.05 s

Selected n_estimators=300 with higher mean CV accuracy: 0.6864 (+/- 0.0568)
Cell 3 completed ✓


In [4]:
# Cell 4 – fit chosen Random Forest on all train rows and evaluate on fold 10 once
rf_best = RandomForestClassifier(n_estimators=best_n, random_state=42, n_jobs=-1, class_weight='balanced')
t0 = time.perf_counter()
rf_best.fit(X_train, y_train)
fit_time = time.perf_counter() - t0
print(f'Fitted RandomForest(n_estimators={best_n}) in {fit_time:.2f} s')

y_pred_rf = rf_best.predict(X_test)

rf_test_acc = float(accuracy_score(y_test, y_pred_rf))
rf_macro_f1 = float(f1_score(y_test, y_pred_rf, average='macro'))
majority_baseline = 0.3240

print(f'\nTest Accuracy        : {rf_test_acc:.4f} ({rf_test_acc*100:.2f}%)')
print(f'Majority Baseline Acc: {majority_baseline:.4f} ({majority_baseline*100:.2f}%)')
print(f'Test Macro-F1        : {rf_macro_f1:.4f}')

print('\n=== Classification Report (Test Fold 10) ===')
report_rf = classification_report(y_test, y_pred_rf, digits=4)
print(report_rf)

# Confusion Matrix
classes = sorted(np.unique(y_test))
cm_rf = confusion_matrix(y_test, y_pred_rf, labels=classes)
cm_rf_df = pd.DataFrame(cm_rf, index=classes, columns=classes)
cm_rf_df.index.name = 'True'
cm_rf_df.columns.name = 'Predicted'

print('=== Confusion Matrix (Fold 10) ===')
print(cm_rf_df.to_string())

fig_dir = REPO_ROOT / 'reports/figures'
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / 'rf_confusion_matrix.png'

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=classes)
disp.plot(ax=ax, cmap='Greens', colorbar=True)
plt.title(f'Random Forest (n={best_n}) - Confusion Matrix (Test Fold 10)')
plt.tight_layout()
plt.savefig(fig_path, dpi=150)
plt.close(fig)
print(f'\nSaved confusion matrix plot to: {fig_path}')

assert rf_test_acc > majority_baseline, f'RF accuracy {rf_test_acc:.4f} did not beat baseline {majority_baseline:.4f}'
print('Cell 4 asserts passed: test accuracy beats majority baseline ✓')

Fitted RandomForest(n_estimators=300) in 3.23 s

Test Accuracy        : 0.7521 (75.21%)
Majority Baseline Acc: 0.3240 (32.40%)
Test Macro-F1        : 0.7178

=== Classification Report (Test Fold 10) ===
                   precision    recall  f1-score   support

     construction     0.7953    0.8724    0.8321       196
             hvac     0.7857    0.6600    0.7174       100
music_loudspeaker     0.6475    0.9000    0.7531       100
            siren     0.8049    0.3976    0.5323        83
     traffic_horn     0.7540    0.7540    0.7540       126

         accuracy                         0.7521       605
        macro avg     0.7575    0.7168    0.7178       605
     weighted avg     0.7620    0.7521    0.7427       605

=== Confusion Matrix (Fold 10) ===
Predicted          construction  hvac  music_loudspeaker  siren  traffic_horn
True                                                                         
construction                171     0                  8      3         


Saved confusion matrix plot to: C:\Users\LENOVO\Desktop\CityPulse\reports\figures\rf_confusion_matrix.png
Cell 4 asserts passed: test accuracy beats majority baseline ✓


In [5]:
# Cell 5 – save candidate RF model, reload, and verify identical predictions
models_dir = REPO_ROOT / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
rf_model_path = models_dir / 'rf_candidate.joblib'
joblib.dump(rf_best, rf_model_path)
print(f'Saved candidate RF model to: {rf_model_path}')

rf_sha = compute_sha256(rf_model_path)
print(f'rf_candidate.joblib sha256: {rf_sha}')

reloaded_rf = joblib.load(rf_model_path)
reloaded_rf_preds = reloaded_rf.predict(X_test)

assert np.array_equal(reloaded_rf_preds, y_pred_rf), 'Reloaded RF model predictions do not equal in-memory predictions'
print('Reloaded model predictions match in-memory predictions exactly ✓')
print('Cell 5 asserts passed ✓')

Saved candidate RF model to: C:\Users\LENOVO\Desktop\CityPulse\models\rf_candidate.joblib
rf_candidate.joblib sha256: 40f38b939c23930f621fa3ba66de0ce06c9f2fec8da70aeaf9b87ad8ed6d6113


Reloaded model predictions match in-memory predictions exactly ✓
Cell 5 asserts passed ✓


In [6]:
# Cell 6 – comparison table: SVM vs Random Forest
svm_model_path = models_dir / 'svm_candidate.joblib'
svm_model = joblib.load(svm_model_path)
y_pred_svm = svm_model.predict(X_test)

svm_test_acc = float(accuracy_score(y_test, y_pred_svm))
svm_macro_f1 = float(f1_score(y_test, y_pred_svm, average='macro'))
svm_report_dict = classification_report(y_test, y_pred_svm, output_dict=True)

svm_cv_csv = REPO_ROOT / 'reports/svm_cv_results.csv'
svm_cv_df = pd.read_csv(svm_cv_csv)
rank_col = 'rank_test_score' if 'rank_test_score' in svm_cv_df.columns else 'rank'
svm_best_cv_row = svm_cv_df.sort_values(rank_col).iloc[0]
svm_cv_mean = float(svm_best_cv_row['mean_test_score'])
svm_cv_std = float(svm_best_cv_row['std_test_score'])

print(f'SVM Fold 10 Accuracy : {svm_test_acc:.4f}')
print(f'SVM CV Accuracy Mean : {svm_cv_mean:.4f}')

assert round(svm_test_acc, 4) == 0.7967, f'Expected SVM fold 10 accuracy 0.7967, got {svm_test_acc:.4f}'
assert round(svm_cv_mean, 4) == 0.6774, f'Expected SVM CV mean 0.6774, got {svm_cv_mean:.4f}'

rf_report_dict = classification_report(y_test, y_pred_rf, output_dict=True)

classes = sorted(np.unique(y_test))
comparison_rows = [
    {
        'model': 'SVM',
        'cv_mean': svm_cv_mean,
        'cv_std': svm_cv_std,
        'fold10_accuracy': svm_test_acc,
        'macro_f1': svm_macro_f1,
        **{f'f1_{cls}': svm_report_dict[cls]['f1-score'] for cls in classes}
    },
    {
        'model': 'RF',
        'cv_mean': best_rf_cv_mean,
        'cv_std': best_rf_cv_std,
        'fold10_accuracy': rf_test_acc,
        'macro_f1': rf_macro_f1,
        **{f'f1_{cls}': rf_report_dict[cls]['f1-score'] for cls in classes}
    }
]

comparison_df = pd.DataFrame(comparison_rows).set_index('model')
print('\n=== Model Comparison: SVM vs Random Forest ===')
print(comparison_df.round(4).to_string())

out_csv = REPO_ROOT / 'reports/svm_vs_rf.csv'
comparison_df.to_csv(out_csv)
print(f'\nSaved comparison table to: {out_csv}')
print('Cell 6 asserts passed ✓')

SVM Fold 10 Accuracy : 0.7967
SVM CV Accuracy Mean : 0.6774

=== Model Comparison: SVM vs Random Forest ===
       cv_mean  cv_std  fold10_accuracy  macro_f1  f1_construction  f1_hvac  f1_music_loudspeaker  f1_siren  f1_traffic_horn
model                                                                                                                       
SVM     0.6774  0.0660           0.7967    0.7780           0.8500   0.6849                0.8406    0.6667           0.8477
RF      0.6864  0.0568           0.7521    0.7178           0.8321   0.7174                0.7531    0.5323           0.7540

Saved comparison table to: C:\Users\LENOVO\Desktop\CityPulse\reports\svm_vs_rf.csv
Cell 6 asserts passed ✓


In [7]:
# Cell 7 – top 15 feature importances of the chosen Random Forest
importances = rf_best.feature_importances_
top15_indices = np.argsort(importances)[::-1][:15]

top15_data = []
for rank, idx in enumerate(top15_indices, 1):
    band_type = 'mean' if idx < 64 else 'std'
    band_num = idx if idx < 64 else idx - 64
    score_imp = float(importances[idx])
    top15_data.append({
        'rank': rank,
        'feature_index': int(idx),
        'band_type': band_type,
        'band_number': int(band_num),
        'importance': score_imp,
        'description': f'Mel band {band_num} ({band_type})'
    })

top15_df = pd.DataFrame(top15_data).set_index('rank')
print('=== Top 15 Feature Importances (Random Forest) ===')
print(top15_df.to_string())
print('\nCell 7 completed ✓')

=== Top 15 Feature Importances (Random Forest) ===
      feature_index band_type  band_number  importance         description
rank                                                                      
1                80       std           16    0.017112   Mel band 16 (std)
2                88       std           24    0.016849   Mel band 24 (std)
3                 0      mean            0    0.016230   Mel band 0 (mean)
4                64       std            0    0.015651    Mel band 0 (std)
5                58      mean           58    0.015319  Mel band 58 (mean)
6                70       std            6    0.014434    Mel band 6 (std)
7                56      mean           56    0.013585  Mel band 56 (mean)
8                57      mean           57    0.013495  Mel band 57 (mean)
9                81       std           17    0.012843   Mel band 17 (std)
10                7      mean            7    0.012609   Mel band 7 (mean)
11               87       std           23    0.0